# SignalDesk Workflow Health Check

**Track A: Fictional Domain Packet**

This notebook helps the SignalDesk product team identify which AI-assisted workflow appears healthiest and which result requires investigation before broader rollout.

The analysis prioritizes human-facing outcomes—completion, output acceptance, review flags, and user ratings—rather than treating model-reported confidence as proof of quality.

In [11]:
import pandas as pd

DATA_PATH = "sample-data/product_usage_events.csv"
df = pd.read_csv(DATA_PATH)

df.head()

,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
0,2026-08-01,Sales,Lead summary,email,42,35,29,3,8.5,0.74,4.1,normal day
1,2026-08-01,Sales,Lead summary,manual,18,12,8,2,6.0,0.61,3.8,normal day
2,2026-08-01,Support,Reply draft,queue,55,48,39,6,4.5,0.82,4.0,normal day
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,14.0,0.59,3.6,small sample


## 1. Data Quality Review

Before calculating workflow metrics, I checked for missing values, inconsistent categories, duplicate export records, and contextual notes that could affect interpretation.

In [12]:
# Review missing values
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

# Review categorical values for inconsistent spelling or casing
for column in ["team", "workflow", "source"]:
    print(f"\n{column}:")
    print(df[column].value_counts())

# Display rows with missing values or non-routine notes
issues = df[
    df.isna().any(axis=1)
    | ~df["notes"].isin(["normal day", "new prompt version started"])
]

display(issues)

Missing values:


,missing_count
date,0
team,0
workflow,0
source,0
sessions,0
completed,0
accepted_output,0
flagged_for_review,0
avg_minutes_saved,0
median_confidence,1



team:
team
Sales      14
Support    13
Product    13
product     1
Name: count, dtype: int64

workflow:
workflow
Lead summary           14
Feedback clustering    14
Reply draft            13
Name: count, dtype: int64

source:
source
manual        19
email          8
queue          7
csv upload     7
Name: count, dtype: int64


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,14.0,0.59,3.6,small sample
5,2026-08-01,Product,Feedback clustering,manual,5,4,3,1,11.0,0.55,3.5,small sample
11,2026-08-02,product,Feedback clustering,manual,6,4,2,1,10.5,0.52,3.4,team casing differs
24,2026-08-05,Sales,Lead summary,email,140,126,119,2,12.0,0.95,4.9,traffic spike from demo account
25,2026-08-05,Sales,Lead summary,email,140,126,119,2,12.0,0.95,4.9,duplicate export row
30,2026-08-05,Product,Feedback clustering,manual,9,7,4,1,11.0,NaN,3.5,confidence missing as text
38,2026-08-07,Support,Reply draft,queue,30,17,8,12,1.5,0.91,2.1,review policy changed mid-day


## 2. Cleaning Decisions

I standardized the inconsistent team capitalization, converted the date column to a datetime type, and removed the row explicitly identified as a duplicate export.

I retained the demo-account traffic row because it represents a real recorded event, but I treat it as non-routine activity when interpreting results. Missing confidence and rating values remain missing rather than being imputed because there is not enough information to estimate them responsibly.

In [13]:
clean_df = df.copy()

# Standardize types and category casing
clean_df["date"] = pd.to_datetime(clean_df["date"])
clean_df["team"] = clean_df["team"].str.strip().str.title()

# Remove only the row explicitly identified as a duplicate export
clean_df = clean_df[
    clean_df["notes"].str.strip().str.lower() != "duplicate export row"
].copy()

print(f"Original rows: {len(df)}")
print(f"Rows after cleaning: {len(clean_df)}")
print(f"Rows removed: {len(df) - len(clean_df)}")

display(
    clean_df[
        clean_df["notes"].isin([
            "traffic spike from demo account",
            "confidence missing as text",
            "review policy changed mid-day"
        ])
    ]
)

Original rows: 41
Rows after cleaning: 40
Rows removed: 1


,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
24,2026-08-05,Sales,Lead summary,email,140,126,119,2,12.0,0.95,4.9,traffic spike from demo account
30,2026-08-05,Product,Feedback clustering,manual,9,7,4,1,11.0,NaN,3.5,confidence missing as text
38,2026-08-07,Support,Reply draft,queue,30,17,8,12,1.5,0.91,2.1,review policy changed mid-day
